# Supervised Classification

In [28]:
from pandas import read_csv
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Daten laden

In [29]:
data = read_csv("https://www.tinyurl.com/mcit-titanic")

## Daten Vertical splitten (Input/Output)

In [30]:
X = data[[
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Embarked'
 ]]

y = data[
    'Survived'
]

## Daten horizontal Splitten (Train/Test)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Daten Vorverarbeiten

Hier haben wir mit Hilfe von Gemini eine Vorverarbeitungspipeline erstellt, die unsere Daten unabhängig vom Datensatz vorbereitet.

In [32]:
num_selector = make_column_selector(dtype_include=['number'])
cat_selector = make_column_selector(dtype_exclude=['number'])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, num_selector),
    ('cat', categorical_transformer, cat_selector)
])

Hier haben wir unsere `preprocessor` Objekt angewendet um unsere eigentlichen Daten zu transformieren. Dazu haben wir den `preprocessor` zuerst mit Hilfe der Traingsdaten eingestellt und anschließend auf unsere Testdaten angewendet. Dieses Vorgehen stellt sicher, dass wir beim Training nicht von unseren Test-Daten lernen. So kann sich weder die Vorverarbeitungspipeline, noch später der ML-Modell einen unfairen Vorteil verschaffen.

In [33]:
preprocessor.fit(X_train)
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

## Model Entwicklung (Machine Learning)

In [36]:
tree_classifier = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=3
)

tree_classifier.fit(X_train_prep, y_train)

print("F1", f1_score(y_test, tree_classifier.predict(X_test_prep)))
print("Recall", recall_score(y_test, tree_classifier.predict(X_test_prep)))
print("Precision", precision_score(y_test, tree_classifier.predict(X_test_prep)))
print("Accuracy", accuracy_score(y_test, tree_classifier.predict(X_test_prep)))

F1 0.6814814814814815
Recall 0.6216216216216216
Precision 0.7540983606557377
Accuracy 0.7597765363128491


In [39]:
print("F1", f1_score(y_train, tree_classifier.predict(X_train_prep)))
print("Recall", recall_score(y_train, tree_classifier.predict(X_train_prep)))
print("Precision", precision_score(y_train, tree_classifier.predict(X_train_prep)))
print("Accuracy", accuracy_score(y_train, tree_classifier.predict(X_train_prep)))

F1 0.7960396039603961
Recall 0.75
Precision 0.8481012658227848
Accuracy 0.8553370786516854


In [38]:
knn_classifier = KNeighborsClassifier(
    n_neighbors=9
)

knn_classifier.fit(X_train_prep, y_train)

print("F1", f1_score(y_test, knn_classifier.predict(X_test_prep)))
print("Recall", recall_score(y_test, knn_classifier.predict(X_test_prep)))
print("Precision", precision_score(y_test, knn_classifier.predict(X_test_prep)))
print("Accuracy", accuracy_score(y_test, knn_classifier.predict(X_test_prep)))

F1 0.59375
Recall 0.5135135135135135
Precision 0.7037037037037037
Accuracy 0.7094972067039106
